# Review Radar — Referenz-Implementierung

**Big Data Analytics (W3-BDA), HTW Berlin — WiSe 2026/2027**

Dieses Notebook ist das *fertige* Produkt, das wir über das Semester gemeinsam
Stück für Stück bauen: aus einem Haufen unstrukturierter Produktbewertungen wird
ein klares Bild — Sentiment, Themen, Zeitverlauf, Zusammenfassung.

Es läuft **out of the box ohne API-Key** (mit einem eingebauten Mock-LLM), damit
ihr die ganze Pipeline sofort seht. Für den echten Betrieb tauscht man eine
Funktion gegen einen echten Anthropic-Aufruf — dazu unten mehr.


## 0. Warum das überhaupt? Das Geschäftsproblem

Ein Unternehmen bekommt jeden Monat tausende Rückmeldungen: Produktbewertungen,
Support-Anfragen, Social-Media-Posts, Umfrage-Kommentare. **Kein Mensch liest das
alles.** Und selbst wer es läse, könnte es nicht verlässlich zu einer Zahl
verdichten, auf die man eine Entscheidung stützt.

Die Kernfrage des Kurses ist deshalb nicht "wie ruft man ein LLM auf", sondern:

> **Wer trifft eine Entscheidung — und was macht diese Person anders,
> weil es dieses Werkzeug gibt?**

Für unser Beispiel: eine Produktmanagerin, die wissen muss, *worüber* sich Kundinnen
beschweren und *ob es schlimmer wird*, um zu entscheiden, was als Nächstes
verbessert wird. Das Dashboard am Ende ist genau für sie gebaut.

Ehrlich bleiben: KI hilft hier echt (Volumen, das niemand von Hand schafft), aber
sie wird auch oft überverkauft. Die eigentliche Kompetenz ist das *Urteil* —
wann lohnt sich das, wann nicht, was kostet es, und darf man dem Ergebnis trauen.
Genau darum dreht sich die zweite Semesterhälfte.


### Dasselbe Muster, viele Anwendungsfälle

Wir bauen *ein* Produkt (Produktbewertungen). Aber das Muster

> **unstrukturierter Text  →  strukturierte Einsicht  →  Entscheidung**

ist überall dasselbe. Was sich ändert, ist nur die Textquelle und die Frage
dahinter:

| Anwendungsfall | Wer entscheidet was? |
|---|---|
| Produktbewertungen *(unser Fall)* | Was verbessern wir als Nächstes? |
| Support-Anfragen | Welche Probleme häufen sich, wo brennt es? |
| Social-Media-Monitoring | Wie kommt eine Kampagne / Marke an? |
| Umfrage-Freitexte | Was sagen Mitarbeitende / Kundinnen wirklich? |
| Produkt-Feedback → Roadmap | Welches Feature wird am meisten gewünscht? |

> ⚠️ *Auch politische Stimmungsanalyse funktioniert technisch genauso — und ist
> genau deshalb ein Warnbeispiel: wessen Meinung wird da eigentlich gemessen,
> und wie leicht ist so etwas verzerrt oder manipuliert? Wir bauen das nicht,
> aber es zeigt, warum Bias und Evaluation (zweite Semesterhälfte) so wichtig
> sind.*

Für die Prüfung nehmt ihr euch **eine** dieser Domänen vor und wendet dasselbe
Werkzeug darauf an — der Beweis, dass ihr das Muster verstanden habt.


## 1. Setup

In Colab braucht ihr nur `anthropic`, wenn ihr den echten LLM-Aufruf nutzt. Für den Mock ist keine Installation nötig.

In [ ]:
# In Colab bei Bedarf:
# !pip install anthropic

import random, json, re
from collections import Counter, defaultdict
from datetime import date, timedelta

## 2. Der Datensatz — synthetische Bewertungen

Statt echter (lizenzrechtlich problematischer) Bewertungen erzeugen wir unseren
eigenen Korpus. Wir *würzen ihn absichtlich* mit dem Chaos, das echte Daten haben:
gemischtes Sentiment, Sarkasmus, Fake-Reviews, ein paar englische, Duplikate, Müll.

Wichtig: wir speichern die **wahre** Kategorie (`true_sentiment`) mit — die
brauchen wir später, um zu *messen*, wie gut das LLM wirklich ist (Gold-Set).

> Produkt: **Nimbus Q2**, ein *fiktiver* Noise-Cancelling-Kopfhörer einer
> erfundenen Marke. Fiktiv = keine echte Marke, keine echten Personen, keine
> Lizenzfragen.

In [ ]:
"""
Synthetic review corpus generator for the BDA course.

Fictional product: the "Nimbus Q2" - a wireless noise-cancelling earbud set
from a made-up brand ("Nimbus Audio"). Fully fictional so there is no real
brand, no real person, and no licensing question - we own this data outright.

The generator deliberately seeds the MESSINESS the course teaches against:
  - mixed sentiment (praise + complaint in one review)
  - sarcasm (positive words, negative meaning)
  - fake / spammy reviews
  - non-German reviews (a few English ones)
  - empty / junk rows and duplicates
  - a rough timeline so sentiment-over-time is analysable

Ground-truth labels ARE recorded here (true_sentiment, is_sarcastic, is_fake)
so later sessions can build a gold set and MEASURE the LLM against it. In the
student-facing corpus we can optionally hide these columns.

This is plain Python + stdlib so it runs anywhere (incl. Colab) with no deps.
"""

import random
import json
import csv
from datetime import date, timedelta

# Deterministic so the corpus is reproducible (matters for teaching + grading)
SEED = 42

PRODUCT = "Nimbus Q2"
BRAND = "Nimbus Audio"

# --- building blocks -------------------------------------------------------

# (text fragment, true sentiment) for genuine positive aspects
POS_ASPECTS = [
    ("Der Klang ist wirklich hervorragend, satte Bässe", "positive"),
    ("Die Geräuschunterdrückung funktioniert im Zug erstaunlich gut", "positive"),
    ("Akku hält locker den ganzen Arbeitstag", "positive"),
    ("Sitzt bequem im Ohr, auch nach Stunden", "positive"),
    ("Verbindung per Bluetooth ist sofort da und stabil", "positive"),
    ("Für den Preis absolut top verarbeitet", "positive"),
]

NEG_ASPECTS = [
    ("Die App stürzt ständig ab", "negative"),
    ("Nach zwei Wochen hat der rechte Ohrhörer aufgehört zu laden", "negative"),
    ("Die Geräuschunterdrückung rauscht hörbar", "negative"),
    ("Viel zu teuer für das, was man bekommt", "negative"),
    ("Die Touch-Steuerung reagiert kaum", "negative"),
    ("Das Case fühlt sich billig und klapprig an", "negative"),
]

NEUTRAL = [
    ("Ganz okay, nichts Besonderes, erfüllt seinen Zweck", "neutral"),
    ("Habe sie seit gestern, kann noch nicht viel sagen", "neutral"),
    ("Standard-Ohrhörer, wie erwartet", "neutral"),
]

# sarcastic: surface sounds positive, true meaning negative
SARCASTIC = [
    "Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
    "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
    "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
    "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
]

# fake / spammy
FAKE = [
    "BESTES PRODUKT EVER!!! Kauft alle bei www.super-deals-guenstig.example!!!",
    "5 Sterne 5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
    "Gratis Gutschein Code NIMBUS100 auf meiner Seite klickt hier jetzt!!!",
    "amazing product best quality buy now discount link in profile",
]

# a few English (multilingual mess)
ENGLISH = [
    ("Sound quality is great but the app is a disaster.", "mixed"),
    ("Battery life is amazing, best earbuds I've owned.", "positive"),
    ("Stopped working after a week, very disappointed.", "negative"),
]

JUNK = ["", "   ", ".", "???", "kein kommentar"]


def _mixed(rng):
    pos, _ = rng.choice(POS_ASPECTS)
    neg, _ = rng.choice(NEG_ASPECTS)
    connector = rng.choice([" - aber ", ", allerdings ", ". Leider "])
    return pos + connector + neg[0].lower() + neg[1:], "mixed"


def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:                      # clean positive
            text, truth = rng.choice(POS_ASPECTS)
        elif r < 0.60:                    # clean negative
            text, truth = rng.choice(NEG_ASPECTS)
        elif r < 0.72:                    # mixed
            text, truth = _mixed(rng)
        elif r < 0.80:                    # neutral
            text, truth = rng.choice(NEUTRAL)
        elif r < 0.88:                    # sarcastic (true = negative)
            text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.94:                    # fake
            text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98:                    # english
            text, truth = rng.choice(ENGLISH)
        else:                             # junk
            text, truth = rng.choice(JUNK), "junk"

        is_sarcastic = text in SARCASTIC
        is_fake = text in FAKE
        # rough timeline across ~9 months, weighted so later months skew worse
        # (simulates a quality dip -> gives sentiment-over-time something to find)
        day_offset = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rating = _rating_for(truth, rng)

        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day_offset)).isoformat(),
            "product": PRODUCT,
            "rating": rating,
            "text": text,
            # --- ground truth (for the gold-set / evaluation sessions) ---
            "true_sentiment": truth,
            "is_sarcastic": is_sarcastic,
            "is_fake": is_fake,
        })
    # a couple of exact duplicates (data-quality session)
    if n > 20:
        rows.append(dict(rows[5], review_id=f"R{n:04d}"))
        rows.append(dict(rows[12], review_id=f"R{n+1:04d}"))
    rng.shuffle(rows)
    return rows


def _rating_for(truth, rng):
    if truth in ("positive",):
        return rng.choice([4, 5, 5])
    if truth in ("negative",):
        return rng.choice([1, 1, 2])
    if truth == "mixed":
        return rng.choice([2, 3, 4])
    if truth == "neutral":
        return 3
    if truth == "fake":
        return 5           # fakes usually 5-star
    return rng.choice([1, 3, 5])  # junk/other: noisy


def save(rows, csv_path, hide_truth=False):
    fields = ["review_id", "date", "product", "rating", "text"]
    if not hide_truth:
        fields += ["true_sentiment", "is_sarcastic", "is_fake"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

In [ ]:
reviews = generate(300)
print(f"{len(reviews)} Bewertungen erzeugt.")
print("Beispiel:")
for r in reviews[:4]:
    print(f"  [{r['rating']}*] {r['text'][:65]}")

## 3. Das LLM-Herzstück

Alles dreht sich um **eine** Funktion: `llm_analyze_review(text)`. Sie bekommt
den Bewertungstext und gibt strukturiert zurück: Sentiment, Themen, Confidence.

Hier ist sie doppelt vorhanden:
- `_mock_llm_analyze` — eine simple, *absichtlich unvollkommene* Attrappe (kein
  Key nötig). Sie liest Sarkasmus wörtlich und erkennt keine Fakes — genau die
  Schwächen, die wir später untersuchen.
- `llm_call_anthropic` — der **echte** Aufruf für Colab.

Über den Schalter `use_real=True` (und einen `client`) wechselt man.

**Merkt euch:** dieses Herzstück ist *nicht* bewertungsspezifisch. Es nimmt Text
und gibt Struktur zurück. Um es auf Tweets oder Support-Tickets zu richten, ändert
man nur den **Prompt** und die **Themen-Liste** — sonst nichts. Genau das macht
das Muster übertragbar.

In [ ]:
import json
import re
from collections import Counter, defaultdict

# ===========================================================================
# LLM LAYER  -  the only part that differs between mock and real
# ===========================================================================

# Canonical theme vocabulary the model is asked to map onto. Keeping a fixed
# set makes aggregation meaningful (free-text themes wouldn't aggregate).
THEMES = ["Klang", "Geräuschunterdrückung", "Akku", "Komfort",
          "Verbindung", "App", "Verarbeitung", "Preis"]

_THEME_HINTS = {
    "Klang": ["klang", "bass", "sound", "bässe"],
    "Geräuschunterdrückung": ["geräuschunterdrückung", "rauscht", "noise"],
    "Akku": ["akku", "laden", "lädt", "battery", "lädt nicht"],
    "Komfort": ["bequem", "ohr", "sitzt", "comfort"],
    "Verbindung": ["bluetooth", "verbindung", "connection"],
    "App": ["app"],
    "Verarbeitung": ["verarbeitet", "case", "billig", "klapprig", "quality", "kaputt"],
    "Preis": ["preis", "teuer", "euro", "price"],
}


def _mock_llm_analyze(text):
    """Fake but plausible LLM output. Deterministic given the text.

    Deliberately imperfect: it reads sarcasm literally (labels sarcastic
    positive-sounding text as positive) and can't tell fakes - exactly the
    failure modes the later course sessions investigate.
    """
    t = (text or "").lower().strip()
    if not t or t in {".", "???", "kein kommentar"}:
        return {"sentiment": "neutral", "themes": [], "confidence": 0.3}

    pos_words = ["hervorragend", "gut", "top", "bequem", "stabil", "satte",
                 "great", "amazing", "best", "super", "toll", "wunderbar",
                 "klasse", "traum", "sagenhafte", "großes kino"]
    neg_words = ["stürzt", "abgestürzt", "aufgehört", "rauscht", "teuer",
                 "kaum", "billig", "klapprig", "kaputt", "disaster",
                 "disappointed", "stopped", "driften", "leider", "nicht"]

    pos = sum(w in t for w in pos_words)
    neg = sum(w in t for w in neg_words)

    # NOTE the naive logic: sarcasm ("super, schon kaputt") counts pos words
    # and often mislabels -> this is intentional teaching material.
    if pos > neg:
        sentiment = "positive"
    elif neg > pos:
        sentiment = "negative"
    else:
        sentiment = "neutral" if (pos == 0 and neg == 0) else "mixed"

    themes = [th for th, hints in _THEME_HINTS.items() if any(h in t for h in hints)]
    confidence = round(min(0.95, 0.5 + 0.1 * abs(pos - neg)), 2)
    return {"sentiment": sentiment, "themes": themes, "confidence": confidence}


def llm_call_anthropic(text, client, model="claude-sonnet-4-5"):
    """REAL call - used in Colab. Returns the same dict shape as the mock.

    Requires: pip install anthropic ; a client created with the group API key.
    Kept here as the drop-in; not executed in the keyless sandbox.
    """
    prompt = f"""Du bist ein Analyse-Tool für Produktbewertungen. Analysiere die
folgende Bewertung und antworte NUR mit JSON, ohne weiteren Text.

Bewertung: \"\"\"{text}\"\"\"

Gib zurück:
{{"sentiment": "positive|negative|neutral|mixed",
  "themes": [aus dieser Liste: {THEMES}],
  "confidence": 0.0-1.0}}"""
    resp = client.messages.create(
        model=model, max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = resp.content[0].text
    return _safe_json(raw)


def _safe_json(raw):
    """Parse model output defensively - it may wrap JSON in prose/backticks.
    (This is the 'reliable structured output' lesson, session 6.)"""
    raw = raw.strip()
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return {"sentiment": "neutral", "themes": [], "confidence": 0.0,
                "_parse_error": True}
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return {"sentiment": "neutral", "themes": [], "confidence": 0.0,
                "_parse_error": True}
    data.setdefault("sentiment", "neutral")
    data.setdefault("themes", [])
    data.setdefault("confidence", 0.0)
    return data


# Toggle here in Colab: use_real=True and pass a client.
def llm_analyze_review(text, use_real=False, client=None):
    if use_real:
        return llm_call_anthropic(text, client)
    return _mock_llm_analyze(text)


# ===========================================================================
# PIPELINE
# ===========================================================================

def classify_all(reviews, **llm_kwargs):
    """Run the LLM over every review, attach structured analysis."""
    out = []
    for r in reviews:
        analysis = llm_analyze_review(r["text"], **llm_kwargs)
        out.append({**r, **analysis})
    return out


def aggregate(analyzed):
    """Turn per-review structured records into the insight layer."""
    sentiments = Counter(a["sentiment"] for a in analyzed)
    theme_counts = Counter()
    theme_by_sentiment = defaultdict(Counter)
    for a in analyzed:
        for th in a["themes"]:
            theme_counts[th] += 1
            theme_by_sentiment[a["sentiment"]][th] += 1

    # sentiment over time (by month)
    by_month = defaultdict(Counter)
    for a in analyzed:
        month = a["date"][:7]  # YYYY-MM
        by_month[month][a["sentiment"]] += 1

    # top complaints = themes most associated with negative reviews
    complaints = theme_by_sentiment["negative"].most_common(5)
    praise = theme_by_sentiment["positive"].most_common(5)

    return {
        "n": len(analyzed),
        "sentiments": dict(sentiments),
        "top_themes": theme_counts.most_common(),
        "top_complaints": complaints,
        "top_praise": praise,
        "by_month": {m: dict(c) for m, c in sorted(by_month.items())},
    }


def executive_summary(agg, product, use_real=False, client=None,
                      model="claude-sonnet-4-5"):
    """One-paragraph summary for a decision-maker.

    use_real=True  -> the LLM writes it from the aggregate numbers (in Colab).
    use_real=False -> deterministic version built from the numbers (keyless).

    The LLM version is deliberately kept: it can *hallucinate* claims the numbers
    don't support, which is exactly what session 9 investigates. Always sanity-
    check an LLM summary against the aggregates.
    """
    if use_real:
        prompt = (
            "Schreibe eine sachliche, einabschnittige Zusammenfassung (Deutsch) "
            "für eine Produktmanagerin, NUR auf Basis dieser Kennzahlen. "
            "Erfinde nichts dazu.\n\n"
            f"Produkt: {product}\n"
            f"Kennzahlen: {json.dumps(agg, ensure_ascii=False)}"
        )
        resp = client.messages.create(
            model=model, max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text.strip()

    n = agg["n"]
    s = agg["sentiments"]
    pos = s.get("positive", 0)
    neg = s.get("negative", 0)
    pct_pos = round(100 * pos / n) if n else 0
    pct_neg = round(100 * neg / n) if n else 0
    top_complaint = agg["top_complaints"][0][0] if agg["top_complaints"] else "-"
    top_praise = agg["top_praise"][0][0] if agg["top_praise"] else "-"
    return (
        f"Von {n} Bewertungen zum {product} sind rund {pct_pos}% positiv und "
        f"{pct_neg}% negativ. Am meisten gelobt wird '{top_praise}', "
        f"häufigster Kritikpunkt ist '{top_complaint}'. "
        f"Die Detailthemen und der zeitliche Verlauf stehen unten."
    )


# ===========================================================================
# DASHBOARD  (clean in-notebook rendering, no chart libs required)
# ===========================================================================

def _bar(count, total, width=24, ch="█"):
    filled = int(round(width * count / total)) if total else 0
    return ch * filled + "·" * (width - filled)


def render_dashboard(agg, product):
    n = agg["n"]
    lines = []
    lines.append("=" * 60)
    lines.append(f"  REVIEW RADAR  -  {product}")
    lines.append(f"  {n} Bewertungen analysiert")
    lines.append("=" * 60)

    lines.append("\n  SENTIMENT")
    order = ["positive", "mixed", "neutral", "negative", "fake", "junk"]
    for s in order:
        c = agg["sentiments"].get(s, 0)
        if c:
            lines.append(f"    {s:9} {_bar(c, n)} {c:3} ({round(100*c/n)}%)")

    lines.append("\n  TOP-LOB")
    for th, c in agg["top_praise"]:
        lines.append(f"    + {th:22} {c}")
    lines.append("\n  TOP-KRITIK")
    for th, c in agg["top_complaints"]:
        lines.append(f"    - {th:22} {c}")

    lines.append("\n  SENTIMENT IM ZEITVERLAUF (pos/neg pro Monat)")
    for month, c in agg["by_month"].items():
        p, ng = c.get("positive", 0), c.get("negative", 0)
        lines.append(f"    {month}   +{p:2}  -{ng:2}   {_bar(p, p+ng+1, 12, '▲')}"
                     f"{_bar(ng, p+ng+1, 12, '▽')}")
    lines.append("=" * 60)
    return "\n".join(lines)

### Den echten Anthropic-Aufruf nutzen (in Colab)

Den Key **niemals** ins Notebook oder ins Repo schreiben. In Colab kommt er aus
dem Secrets-Panel (Schlüssel-Symbol links), Name z. B. `ANTHROPIC_API_KEY`:

```python
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# dann in der Pipeline:  classify_all(reviews, use_real=True, client=client)
```

Solange `use_real=False` bleibt, läuft alles über den Mock — key-frei.

## 4. Klassifizieren — über den ganzen Korpus

Ein LLM-Aufruf pro Bewertung, Ergebnisse an den Datensatz angehängt.

In [ ]:
analyzed = classify_all(reviews, use_real=False)   # use_real=True für echt
print("Analysiert:", len(analyzed))
print(json.dumps({k: analyzed[0][k] for k in ("text","sentiment","themes","confidence")},
                 ensure_ascii=False, indent=2))

## 5. Aggregieren — vom Einzelfall zum Gesamtbild

Aus hunderten Einzelurteilen werden Kennzahlen: Sentiment-Verteilung, Top-Themen, Lob vs. Kritik, Verlauf über die Zeit.

In [ ]:
agg = aggregate(analyzed)
print("Sentiment:", agg["sentiments"])
print("Top-Kritik:", agg["top_complaints"][:3])

## 6. Executive Summary + Dashboard

Das, was eine Entscheiderin in 30 Sekunden liest. Die Zusammenfassung schreibt
das **LLM** aus den Aggregat-Zahlen (mit Mock als Fallback ohne Key).

> Kleiner Vorgriff auf später: eine LLM-geschriebene Zusammenfassung kann Dinge
> behaupten, die in den Zahlen gar nicht stehen (*Halluzination*). Genau deshalb
> prüfen wir sie in Abschnitt 7 gegen die echten Daten.

In [ ]:
summary = executive_summary(agg, "Nimbus Q2", use_real=False)
print(summary)
print()
print(render_dashboard(agg, "Nimbus Q2"))

## 7. Und jetzt das Interessante: stimmt das überhaupt?

Das Dashboard *sieht* überzeugend aus. Aber:
- Wie viele Bewertungen hat das LLM **falsch** eingeordnet?
- Was macht es mit **Sarkasmus** ("Super, schon kaputt.")?
- Was mit **Fake-Reviews**?

Weil wir die wahre Kategorie mitgespeichert haben, können wir das *messen* —
das ist der rote Faden der zweiten Semesterhälfte.

In [ ]:
comparable = [a for a in analyzed if a["true_sentiment"] in
              ("positive","negative","neutral","mixed")]
correct = sum(1 for a in comparable if a["sentiment"] == a["true_sentiment"])
print(f"Übereinstimmung mit Wahrheit: {correct}/{len(comparable)} "
      f"= {round(100*correct/len(comparable))}%")

sarc = [a for a in analyzed if a["is_sarcastic"]]
sarc_wrong = sum(1 for a in sarc if a["sentiment"] != "negative")
print(f"Sarkasmus falsch gelabelt: {sarc_wrong}/{len(sarc)}")

fake = [a for a in analyzed if a["is_fake"]]
print("Fakes gelabelt als:", dict(Counter(a["sentiment"] for a in fake)))

---
*Code: MIT · Materialien: CC BY 4.0 · (c) 2026 Kevin Metka.*
*Der Mock ist deterministisch; der echte LLM-Aufruf ist es nicht — dieselbe
Bewertung kann zweimal unterschiedlich ausfallen. Auch das ist Thema im Kurs.*